# Week 5 — Subqueries and Window Functions: Subqueries
## Phase 2b SQL | PORA Academy Cohort 7 — **Exercises**

Wednesday's demo used one query's result to drive another — a threshold pulled live from `AVG(payment_value)`, a whole set of `customer_id`s pulled from a `WHERE customer_state = 'SP'` filter, a derived table built from a `GROUP BY` you couldn't otherwise fold into one aggregate. Today you write those same three shapes yourself, and then combine them: several scalar subqueries side by side in one row, a derived table you build and immediately divide, and a `NOT IN` you reach for on purpose instead of by accident.

Each question below comes as **three cells**:

1. A **question** with the task and an **Expected** result.
2. A blank `%%sql` answer cell — write your query where it says `-- Your query here`, capturing the result into a variable (e.g. `q1`).
3. A **check cell** (plain Python) — run it after your query. A ✅ means you got it right, and the cell then displays the table your query returned.

**Do not edit the check cells.** Run the setup cell first, then work top to bottom. The check cells read the exact column aliases each question asks for, so use the alias names as written.

🤖 **Using DeepSeek this week:** you may ask DeepSeek to help draft these queries, but the prompt-then-verify protocol from the demo still applies — tell it the table(s), the exact columns, and that the dialect is SQLite, then **run the query and check the number against the Expected value before you trust it**. Subqueries are an easy place for AI-drafted SQL to look plausible and be subtly wrong — a `=` where a subquery can return more than one row, or a `NOT IN` against a list that secretly contains a `NULL` — so the check cells are the verification step; never edit one to make a wrong query pass.

In [1]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


Mounted at /content/drive
Searching your Google Drive for phase-2-python-sql.zip ...
Unzipping phase-2-python-sql.zip ...
Data folder: /content/olist_data/phase-2-python-sql
Loaded orders: 99,441 rows
Loaded customers: 99,441 rows
Loaded order_items: 112,650 rows
Loaded order_payments: 103,886 rows
Loaded order_reviews: 99,224 rows
Loaded products: 32,951 rows
Loaded sellers: 3,095 rows
Loaded product_category_translation: 71 rows

Database ready.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.1/615.1 kB 28.4 MB/s eta 0:00:00


## Question 1 — The threshold and the count, in one query

Concept 1 in the demo split this into two cells: one query found the average `payment_value`, a second counted how many payments cleared it. Do both in a **single** query against `order_payments`, using a scalar subquery in `SELECT` and the same scalar subquery again in `WHERE`:

- `avg_payment` — a scalar subquery in the `SELECT` list: `(SELECT ROUND(AVG(payment_value), 2) FROM order_payments)`
- `high_value_count` — `COUNT(*)`, filtered by `WHERE payment_value > (SELECT AVG(payment_value) FROM order_payments)`

**Expected:** one row — `avg_payment` = R$154.10, `high_value_count` = 31,012.

In [11]:
%%sql q1 <<
SELECT
    (SELECT ROUND(AVG(payment_value), 2)
     FROM order_payments) AS avg_payment,
    COUNT(*) AS high_value_count
FROM order_payments
WHERE payment_value > (
    SELECT AVG(payment_value)
    FROM order_payments
);

In [12]:
# --- CHECK Q1 — do not edit ---
for col in ['avg_payment', 'high_value_count']:
    assert col in q1.columns, f"Q1: missing the '{col}' column — check your SELECT aliases"
assert q1.shape[0] == 1, f"Q1: expected a single row, got {q1.shape[0]}"
assert abs(float(q1.iloc[0]['avg_payment']) - 154.10) < 0.01, \
    f"Q1: expected avg_payment ≈ 154.10, got {q1.iloc[0]['avg_payment']}"
assert int(q1.iloc[0]['high_value_count']) == 31012, \
    f"Q1: expected high_value_count = 31,012, got {int(q1.iloc[0]['high_value_count']):,}"
print("✅ Q1 correct")
q1  # show the result of your query

✅ Q1 correct


,avg_payment,high_value_count
0,154.1,31012


## Question 2 — `IN` subquery, plus a scalar subquery for context

Concept 3 counted delivered orders from São Paulo customers with `customer_id IN (SELECT customer_id FROM customers WHERE customer_state = 'SP')`. Write that same `IN` subquery against `orders`, and in the same `SELECT` list add a second, independent scalar subquery reporting how many customers are based in SP in total — so the reader can see the delivered count next to the population it was drawn from.

Return one row with two columns:

- `sp_customer_count` — a scalar subquery: `(SELECT COUNT(*) FROM customers WHERE customer_state = 'SP')`
- `sp_delivered_orders` — `COUNT(*)` from `orders`, filtered to `order_status = 'delivered'` **and** the `IN` subquery above

**Expected:** one row — `sp_customer_count` = 41,746, `sp_delivered_orders` = 40,501.

In [4]:
%%sql q2 <<
SELECT
    (SELECT COUNT(*) FROM customers WHERE customer_state = 'SP') AS sp_customer_count,
    COUNT(*) AS sp_delivered_orders
FROM orders
WHERE order_status = 'delivered'
  AND customer_id IN (
      SELECT customer_id
      FROM customers
      WHERE customer_state = 'SP'
  );

In [5]:
# --- CHECK Q2 — do not edit ---
for col in ['sp_customer_count', 'sp_delivered_orders']:
    assert col in q2.columns, f"Q2: missing the '{col}' column — check your SELECT aliases"
assert q2.shape[0] == 1, f"Q2: expected a single row, got {q2.shape[0]}"
assert int(q2.iloc[0]['sp_customer_count']) == 41746, \
    f"Q2: expected sp_customer_count = 41,746, got {int(q2.iloc[0]['sp_customer_count']):,}"
assert int(q2.iloc[0]['sp_delivered_orders']) == 40501, \
    f"Q2: expected sp_delivered_orders = 40,501, got {int(q2.iloc[0]['sp_delivered_orders']):,}"
print("✅ Q2 correct")
q2  # show the result of your query

✅ Q2 correct


,sp_customer_count,sp_delivered_orders
0,41746,40501


## Question 3 — A derived table, then a percentage (mind the integer division)

Build on the "subquery in `FROM`" pattern from Concept 2. Write a query with a derived table (a subquery inside `FROM`) that has exactly two columns, each built from its own scalar subquery:

- `null_category_products` — a scalar subquery counting `products` where `product_category_name IS NULL`
- `total_products` — a scalar subquery counting every row in `products`

Then, in the **outer** query, `SELECT` both columns from the derived table plus a third column, `null_pct` — `null_category_products` as a percentage of `total_products`, rounded to 2 decimal places. Force real division: `* 100.0`, not `* 100`.

**Expected:** one row — `null_category_products` = 610, `total_products` = 32,951, `null_pct` ≈ 1.85. A value of 1 or 0 for `null_pct` means integer division silently truncated it.

In [15]:
%%sql q3 <<
SELECT
    null_category_products,
    total_products,
    ROUND(null_category_products * 100.0 / total_products, 2) AS null_pct
FROM (
    SELECT
        (SELECT COUNT(*) FROM products WHERE product_category_name IS NULL) AS null_category_products,
        (SELECT COUNT(*) FROM products) AS total_products
);

In [16]:
# --- CHECK Q3 — do not edit ---
for col in ['null_category_products', 'total_products', 'null_pct']:
    assert col in q3.columns, f"Q3: missing the '{col}' column — check your SELECT aliases"
assert q3.shape[0] == 1, f"Q3: expected a single row, got {q3.shape[0]}"
assert int(q3.iloc[0]['null_category_products']) == 610, \
    f"Q3: expected null_category_products = 610, got {int(q3.iloc[0]['null_category_products']):,}"
assert int(q3.iloc[0]['total_products']) == 32951, \
    f"Q3: expected total_products = 32,951, got {int(q3.iloc[0]['total_products']):,}"
got_pct = float(q3.iloc[0]['null_pct'])
assert abs(got_pct - 1.85) < 0.01, \
    (f"Q3: expected null_pct ≈ 1.85, got {got_pct} — a value of 1 or 0 means integer "
     f"division truncated it; multiply by 100.0, not 100")
print("✅ Q3 correct")
q3  # show the result of your query

✅ Q3 correct


,null_category_products,total_products,null_pct
0,610,32951,1.85


## Question 4 — `NOT IN`, used on purpose

`IS NULL` is the direct way to find products with a missing `product_weight_g` (Concept 3 already showed there are 2 of them). Find the same 2 products a different way: write a query against `products` that counts rows using `product_id NOT IN (subquery)`, where the subquery selects the `product_id` of every product whose `product_weight_g` **is not** `NULL`.

Alias the count as `null_weight_count`.

Because the inner subquery explicitly filters out `NULL` weights, its list of `product_id`s can never itself contain a `NULL` — which is exactly what makes `NOT IN` safe here. (Skip that filter and the "Going deeper" trap from the demo applies: a `NOT IN` list containing even one `NULL` silently matches zero rows.)

**Expected:** `null_weight_count` = 2.

In [17]:
%%sql q4 <<
SELECT COUNT(*) AS null_weight_count
FROM products
WHERE product_id NOT IN (
    SELECT product_id
    FROM products
    WHERE product_weight_g IS NOT NULL
);

In [18]:
# --- CHECK Q4 — do not edit ---
assert 'null_weight_count' in q4.columns, "Q4: missing the 'null_weight_count' column — check your SELECT alias"
assert q4.shape[0] == 1, f"Q4: expected a single row, got {q4.shape[0]}"
assert int(q4.iloc[0]['null_weight_count']) == 2, \
    (f"Q4: expected null_weight_count = 2, got {int(q4.iloc[0]['null_weight_count']):,} — "
     f"if you got 0, your NOT IN list likely contains a NULL (a classic subquery trap)")
print("✅ Q4 correct")
q4  # show the result of your query

✅ Q4 correct


,null_weight_count
0,2


## Question 5 — The complement: delivered orders outside SP

Question 2 counted the 40,501 delivered orders **from** SP customers. Now write the complement: count delivered orders (`orders.order_status = 'delivered'`) whose `customer_id` is `NOT IN` the same subquery you used in Question 2 (customers where `customer_state = 'SP'`).

Alias the count as `non_sp_delivered`.

**Expected:** `non_sp_delivered` = 55,977 (96,478 total delivered orders − 40,501 delivered from SP — both counts you've already verified today).

In [19]:
%%sql q5 <<
SELECT COUNT(*) AS non_sp_delivered
FROM orders
WHERE order_status = 'delivered'
  AND customer_id NOT IN (
      SELECT customer_id
      FROM customers
      WHERE customer_state = 'SP'
  );

In [20]:
# --- CHECK Q5 — do not edit ---
assert 'non_sp_delivered' in q5.columns, "Q5: missing the 'non_sp_delivered' column — check your SELECT alias"
assert q5.shape[0] == 1, f"Q5: expected a single row, got {q5.shape[0]}"
# Derived from two verified counts: 96,478 delivered orders total minus 40,501 delivered from SP.
expected = 96478 - 40501
assert int(q5.iloc[0]['non_sp_delivered']) == expected, \
    f"Q5: expected non_sp_delivered = {expected:,}, got {int(q5.iloc[0]['non_sp_delivered']):,}"
print("✅ Q5 correct")
q5  # show the result of your query

✅ Q5 correct


,non_sp_delivered
0,55977
